# The Cost Function
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** what a cost function measures and why minimizing it is the same as training a model
- **Describe** the shape of the MSE cost surface and why it has exactly one minimum
- **Interpret** the connection between moving the regression line and moving along the cost surface

> **Tip:** Start by moving the **slope and intercept sliders** to fit a line to the scatter plot, and watch the MSE counter decrease as the line gets closer to the data. The moment the MSE stops decreasing is the moment gradient descent would stop.

---
## How we got here

In *06: Gradient & Gradient Descent* we animated a ball rolling down a loss surface. In *05: Partial Derivatives* we computed the gradient of a loss with respect to multiple parameters. This notebook puts the two together and asks: what exactly is that surface we are rolling down? The cost function is the answer. It is the mathematical object that translates "how wrong is the model" into a number that calculus can minimize.

---
## Why this matters for data science

Choosing the right cost function is one of the most impactful design decisions when building a model. MSE penalizes large errors heavily because it squares them. MAE treats all errors equally. Cross-entropy penalizes confident wrong predictions much more than uncertain ones. Each choice shapes the loss surface differently, which changes what the model learns to prioritize. Understanding cost functions means understanding what your model is actually optimizing for.

---
## Try it yourself

In [ ]:
np.random.seed(42)
x_data = np.linspace(0, 10, 25)
y_data = 2.5 * x_data + 4 + np.random.normal(0, 3, 25)

m_range = np.linspace(-1.0, 6.0, 60)
b_range = np.linspace(-10.0, 20.0, 60)
cost_surface = np.array([[np.mean((y_data - (m * x_data + b))**2)
                           for m in m_range] for b in b_range])

out_scatter = Output()
out_surface = Output()
out_3d      = Output()

m_slider = FloatSlider(value=0.0, min=-1.0, max=6.0, step=0.1,
    description="Slope (m):",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"))

b_slider = FloatSlider(value=0.0, min=-10.0, max=20.0, step=0.5,
    description="Intercept (b):",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"))

x_line = np.linspace(x_data.min() - 0.5, x_data.max() + 0.5, 100)

def render(m, b):
    y_hat = m * x_data + b
    mse   = np.mean((y_data - y_hat) ** 2)

    # ── Scatter plot with residuals ──────────────────────────────────────
    x_resid, y_resid = [], []
    for xi, yi, yfi in zip(x_data, y_data, y_hat):
        x_resid += [xi, xi, None]
        y_resid += [yi, yfi, None]

    traces1 = [
        go.Scatter(x=x_resid, y=y_resid, mode="lines",
            line=dict(color="orange", width=1.5), opacity=0.75,
            name="Residuals", showlegend=False),
        go.Scatter(x=x_data, y=y_data, mode="markers",
            marker=dict(color=PALETTE["muted"], size=8, opacity=0.8), name="Data"),
        go.Scatter(x=x_line, y=m * x_line + b, mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5),
            name=f"y = {m:.2f}x + {b:.2f}"),
    ]
    layout1 = base_layout(
        title=f"y = {m:.2f}x + {b:.2f}<br>MSE = {mse:.2f}",
        xaxis_title="x", yaxis_title="y")
    with out_scatter:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces1, layout=layout1)))

    # ── 2D contour map ───────────────────────────────────────────────────
    traces2 = [
        go.Contour(z=cost_surface, x=m_range, y=b_range,
            colorscale="Blues", ncontours=25, showscale=False, name="MSE surface"),
        go.Scatter(x=[m], y=[b], mode="markers",
            marker=dict(color=PALETTE["secondary"], size=14, symbol="star"),
            name=f"MSE = {mse:.2f}"),
    ]
    layout2 = base_layout(
        title="MSE Cost Surface (top-down view) — star marks your current (m, b)",
        xaxis_title="Slope (m)", yaxis_title="Intercept (b)")
    with out_surface:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces2, layout=layout2)))

    # ── 3D surface ───────────────────────────────────────────────────────
    traces3 = [
        go.Surface(x=m_range, y=b_range, z=cost_surface,
            colorscale="Blues", opacity=0.85, showscale=False,
            name="MSE surface"),
        go.Scatter3d(x=[m], y=[b], z=[mse], mode="markers",
            marker=dict(color=PALETTE["secondary"], size=7, symbol="diamond"),
            name=f"Current: MSE = {mse:.2f}"),
    ]
    layout3 = go.Layout(
        title=dict(
            text="MSE Cost Surface (3D) — one bowl, one minimum",
            font=dict(size=FONT["size_title"]), x=0.5),
        paper_bgcolor=PALETTE["background"],
        scene=dict(
            xaxis=dict(title="Slope (m)",     backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
            yaxis=dict(title="Intercept (b)", backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
            zaxis=dict(title="MSE",           backgroundcolor=PALETTE["background"], gridcolor="#DEE2E6"),
            bgcolor=PALETTE["background"],
        ),
        height=480,
        showlegend=False,
    )
    with out_3d:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces3, layout=layout3)))

def on_change(change): render(m_slider.value, b_slider.value)
m_slider.observe(on_change, names="value")
b_slider.observe(on_change, names="value")
display(VBox([m_slider, b_slider, out_scatter, out_surface, out_3d]))
render(0.0, 0.0)

---
## What's happening?

A cost function (also called a loss function) takes the model's predictions and the true labels, compares them, and produces a single number measuring how wrong the model is. Minimizing that number is training.

The most common cost function for regression is **Mean Squared Error (MSE)**:

$$\text{MSE}(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2$$

Here n is the number of training examples, y_i is the true value, and ŷ_i is the model's prediction. The difference (y_i − ŷ_i) is the **residual**, how far off the prediction was. Squaring makes all residuals positive and penalizes large errors more than small ones.

| Cost function | Formula (simplified) | What it penalizes | Which models use it |
|--------------|---------------------|-------------------|---------------------|
| MSE | mean of (y − ŷ)² | Large errors heavily | Linear regression, neural networks (regression) |
| MAE | mean of |y − ŷ| | All errors equally | Robust regression, some neural networks |
| Cross-entropy | −mean of y·log(ŷ) | Confident wrong predictions most | Logistic regression, classification neural nets |
| Hinge loss | mean of max(0, 1 − y·ŷ) | Margin violations | Support vector machines |

### Why MSE is a bowl

For linear regression, MSE as a function of the weights is always a convex quadratic, a bowl shape with no local minima, just one global minimum. This is a very special property. It means gradient descent is guaranteed to find the best solution, no matter where it starts. Many more complex models do not have this guarantee.

Move both sliders in the widget to the position that minimizes MSE. Compare that position to the lowest point of the cost surface contour map.

---
## Real-world example: MSE vs MAE on a salary prediction dataset

The choice between MSE and MAE changes which errors the model focuses on. MSE's squaring means that one very large error (a salary misforecast of $200k) dominates the loss calculation and pulls the fitted line toward that outlier. MAE treats a $200k error as 200 times a $1k error, not 40,000 times.

The chart shows both cost functions evaluated across the same weight grid for a salary prediction problem with one deliberate outlier.

Notice:

- **Notice:** The MSE surface (left) has its minimum pulled toward the outlier; the fitted line passes closer to the outlier point at the expense of fitting the bulk of the data less well
- **Notice:** The MAE surface (right) has a flatter minimum region and its fitted line better represents the typical data point; the outlier has less influence
- **Notice:** MAE's cost surface has a sharp ridge at the minimum (not smooth), which makes it harder to minimize with gradient descent; MSE's smooth bowl is easier to optimize

> **Discussion question:** You are predicting house prices and your dataset has five data-entry errors where prices are 10x higher than they should be. Would you use MSE or MAE for training, and what else might you do before choosing a loss function?

In [3]:
np.random.seed(42)
x_data = np.linspace(0, 10, 40)
y_clean = 3.0 * x_data + 5 + np.random.normal(0, 3, 40)
y_data  = y_clean.copy()
y_data[38] = 80   # deliberate outlier

w_range = np.linspace(-1, 7, 60)
b_range = np.linspace(-10, 20, 60)
W, B = np.meshgrid(w_range, b_range)

mse_surface = np.array([[np.mean((y_data - (w * x_data + b))**2)
                          for w in w_range] for b in b_range])
mae_surface = np.array([[np.mean(np.abs(y_data - (w * x_data + b)))
                          for w in w_range] for b in b_range])

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["MSE Cost Surface — pulled toward outlier",
                    "MAE Cost Surface — more robust to outlier"],
    specs=[[{"type":"contour"},{"type":"contour"}]])
for col, (Z, label) in enumerate([(mse_surface, "MSE"), (mae_surface, "MAE")], start=1):
    fig.add_trace(go.Contour(z=Z, x=w_range, y=b_range,
        colorscale="Blues" if col==1 else "Greens",
        showscale=False, ncontours=20, name=label), row=1, col=col)
    w_opt = w_range[np.argmin(Z.min(axis=0))]
    b_opt = b_range[np.argmin(Z.min(axis=1))]
    fig.add_trace(go.Scatter(x=[w_opt], y=[b_opt], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=12, symbol="star"),
        name=f"{label} minimum"), row=1, col=col)
fig.update_layout(
    title=dict(text="MSE vs MAE Cost Surfaces — Same Data, Different Objectives",
               font=dict(size=FONT["size_title"])),
    paper_bgcolor=PALETTE["background"], height=400, showlegend=False)
fig.update_xaxes(title_text="Weight w", gridcolor="#DEE2E6")
fig.update_yaxes(title_text="Intercept b", gridcolor="#DEE2E6")
fig.show()

### Cost function comparison guide

| Cost function | Formula | What it penalizes | Which ML models use it |
|--------------|---------|-------------------|----------------------|
| MSE | mean(y − ŷ)² | Large errors very heavily (squares them) | Linear regression, regression neural networks |
| RMSE | sqrt(MSE) | Same as MSE but in original units | Reporting metric; same minimum as MSE |
| MAE | mean(|y − ŷ|) | All errors equally | Robust regression; median prediction |
| Cross-entropy | −mean(y·log(ŷ)) | Confident wrong predictions severely | Logistic regression, classification neural networks |
| Hinge | mean(max(0, 1−y·ŷ)) | Margin violations only | Support vector machines |

---
## Key takeaway

> **The cost function translates model predictions into a single number measuring how wrong the model is; minimizing that number over the training data is what model training literally is, and the shape of the cost surface determines how hard that minimization is.**

---
*Next up: Optimization — why some loss surfaces are easy to minimize and others are not, and what techniques help gradient descent succeed on the hard ones*